# Galerie modèles pré-entraînés — [02 🟢] Tri de tickets sans labels (CapGroup)

> Optionnel, pas de livrable. Geste : **classification zero-shot** — classer **sans aucune donnée d'entraînement**.

**Contexte fictif.** *CapGroup* (le client du cas M8) reçoit ~80 tickets RH/jour à trier en 5 catégories. Problème : **pas encore d'historique labellisé**. Un modèle **zero-shot** permet de classer en définissant simplement les **labels candidats** — sans entraînement.

On suit le même **pattern en 4 temps** qu'au notebook 01.

## Setup

> ⚠️ **Premier lancement = téléchargement du modèle** (connexion requise une fois, puis mise en cache locale). Tout tourne sur **CPU**, pas besoin de GPU.

```bash
pip install "transformers>=4.40" torch sentence-transformers scikit-learn pandas
```

> ⚠️ **Piège réel** : le modèle zero-shot multilingue utilise un tokenizer *SentencePiece*. Si tu obtiens une erreur `SentencePiece library ... not found` ou `protobuf ... not found`, installe :
> ```bash
> pip install sentencepiece protobuf
> ```

In [ ]:
import time
from transformers import pipeline

## [1] Choisir le modèle

- **Modèle** : `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`
- **Tâche** : `zero-shot-classification` (NLI multilingue)
- **Langue** : multilingue, **français** ✅
- **Licence** : MIT ✅
- **Taille** : ~560 Mo


## [2] Charger via `pipeline()`

In [ ]:
MODELE = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
t0 = time.perf_counter()
zs = pipeline("zero-shot-classification", model=MODELE)
print(f"Modèle chargé en {time.perf_counter()-t0:.1f}s")

## [3] Classer des tickets — sans entraînement

On définit nos **5 catégories** comme labels candidats.

In [ ]:
CATEGORIES = ["paie", "mutuelle", "congés", "formation", "autre"]
tickets = [
    "Mon prélèvement de mutuelle est erroné ce mois-ci",
    "Je voudrais poser mes congés pour août",
    "Quand sera versé mon salaire de juin ?",
]
for t in tickets:
    out = zs(t, candidate_labels=CATEGORIES)
    print(f"{out['labels'][0]:>10} ({out['scores'][0]:.2f})  | {t}")

**Résultat attendu** : `mutuelle` / `congés` / `paie`. Le modèle **n'a jamais vu** de ticket CapGroup : il raisonne sur le **sens** des labels. C'est la magie — et la limite — du zero-shot.


## [4] Sobriété — zero-shot vs supervisé

Le zero-shot est **idéal au démarrage à froid** (aucun label disponible). Mais dès que CapGroup aura accumulé un **historique labellisé** (ce qui est le cas dans le brief M8 : ~10 000 tickets catégorisés), un **classifieur supervisé** (TF-IDF + régression logistique, ou un SLM fine-tuné léger) sera :
- **plus précis** (entraîné sur le vrai vocabulaire interne),
- **bien plus léger et rapide** (~0 Mo vs ~560 Mo),
- **explicable**.


## 🤔 Question réflexive

Le zero-shot évite l'entraînement, mais charge un gros modèle généraliste à chaque inférence. 

> 💡 **Règle** : *si tu as un dataset labellisé, le supervisé bat généralement le zero-shot* — en précision **et** en coût. Le zero-shot brille quand tu n'as **pas** de labels. C'est exactement l'arbitrage du brief **M8-B2** (`05_Zero_shot_suffit.md`).


## 📝 Verdict — à formuler en 3 lignes

À quelle condition recommanderais-tu de **rester** en zero-shot pour CapGroup, et à quel moment **basculer** vers un supervisé ? Écris ton verdict.